# Document Image Segmentation

This notebook trains a U-Net model for **pixel-level document segmentation** using DocBank data converted to segmentation masks.

**NOTE:** For this demonstration, we are only using a small sample from the DocBank dataset. However, you can use the entire dataset to train a full segmentation model.

To know more about DocBank Dataset you can visit: https://github.com/doc-analysis/DocBank

## Pre-requisites

To support features of this notebook with CoreAI, we need to install some libraries that are not pre-installed but are required for this notebook. 

## Create and Activate the Virtual Environment:
Open your terminal or command prompt within the Jupyter notebook. Navigate via `File -> New -> Terminal`.
Type `bash` to access a shell compatible with the following commands.
Navigate to the project directory where you want to set up the environment (where this notebook is located):

```bash
export PROJECT_NAME="Doc_Image_Seg"
export PIP_CACHE_DIR=`pwd`/.cache/pip
mkdir -p $PIP_CACHE_DIR
python -m venv --system-site-packages myvenv
source myvenv/bin/activate
pip install ipykernel
python -m ipykernel install --user --name=${PROJECT_NAME}_myvenv --display-name="Python (${PROJECT_NAME}_myvenv)"
echo ""; echo "Before continuing load the created Python kernel: Python (${PROJECT_NAME}_myvenv)"
```

Load the Python kernel described above before running the cell below (it might take a few seconds for the kernel to appear in the list of kernels).

The following will set the folder location for download so that they are local to the running container, to provide cache.

In [ ]:
import os
def set_env_with_cache_dir(env_var_name: str, subdir: str):
    base_cache = os.path.join(os.getcwd(), ".cache")
    full_path = os.path.join(base_cache, subdir)
    os.environ[env_var_name] = full_path
    os.makedirs(full_path, exist_ok=True)
    print(f"{env_var_name}={full_path}")

set_env_with_cache_dir("PIP_CACHE_DIR", "pip")
set_env_with_cache_dir("HF_HOME", "huggingface")
set_env_with_cache_dir("TORCH_HOME", "torch")

#### Cloning the DockBank Repository 

In [ ]:
!git clone https://github.com/doc-analysis/DocBank.git

#### Creating the Dataset for Pixel Segmentation 

In [ ]:
if not os.path.exists("pixel_segmentation_dataset"):
    os.system("python convert.py")
else:
    print("The dataset already exists.")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torch.nn.functional as F
import json
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import cv2


DOCBANK_LABELS = [ 'background', 'abstract', 'author', 'caption', 'date', 'equation', 'figure', 'footer', 'list', 'paragraph', 'reference', 'section', 'table', 'title']

NUM_CLASSES = len(DOCBANK_LABELS)

LABEL_COLORS = {
    'background': (0, 0, 0),          # Black
    'abstract': (255, 0, 0),          # Red
    'author': (0, 255, 0),            # Green  
    'caption': (0, 0, 255),           # Blue
    'date': (255, 255, 0),            # Yellow
    'equation': (255, 0, 255),        # Magenta
    'figure': (0, 255, 255),          # Cyan
    'footer': (128, 128, 128),        # Gray
    'list': (255, 165, 0),            # Orange
    'paragraph': (128, 0, 128),       # Purple
    'reference': (255, 192, 203),     # Pink
    'section': (0, 128, 0),           # Dark Green
    'table': (165, 42, 42),           # Brown
    'title': (0, 0, 128),             # Navy
}

print(f" CUDA available: {torch.cuda.is_available()}")
print(f" Number of classes: {NUM_CLASSES}")
print(f" Classes: {DOCBANK_LABELS}")

The`DocumentSegmentationDataset` class is designed for loading paired document images and their corresponding segmentation masks, which is commonly used in document layout analysis or image segmentation tasks.

In [ ]:
class DocumentSegmentationDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None, mask_transform=None):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.transform = transform
        self.mask_transform = mask_transform
        self.image_files = list(self.images_dir.glob("*.jpg")) + list(self.images_dir.glob("*.png"))
        
        valid_files = []
        for img_file in self.image_files:
            mask_file = self.masks_dir / f"{img_file.stem}_mask.png"
            if mask_file.exists():
                valid_files.append(img_file)
        
        self.image_files = valid_files
        print(f"Found {len(self.image_files)} valid image-mask pairs")
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert('RGB')
        
        mask_path = self.masks_dir / f"{img_path.stem}_mask.png"
        mask = Image.open(mask_path).convert('L')  
        
        img_size = image.size 
        mask_size = mask.size
        if img_size != mask_size:
            print(f"Size mismatch for {img_path.name}: Image {img_size} vs Mask {mask_size}")
            mask = mask.resize(img_size, Image.NEAREST)
            print(f"Fixed: Resized mask to {mask.size}")
        
        mask_array = np.array(mask)
        
        if self.transform:
            image = self.transform(image)
            target_size = image.shape[-2:] 
            mask_tensor = torch.from_numpy(mask_array).unsqueeze(0).float() 
            mask = F.interpolate(
                mask_tensor.unsqueeze(0), 
                size=target_size, 
                mode='nearest'
            ).squeeze().long()
        else:
            mask = torch.from_numpy(mask_array).long()
        
        return image, mask

This function defines and returns data transformation pipelines for training and validation datasets in a segmentation task.

In [ ]:
def create_data_transforms(image_size=512):
    train_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    return train_transform, val_transform

### UNet Model for Image Segmentation

This class defines a U-Net model, a popular convolutional neural network architecture for image segmentation tasks. It takes an image as input and outputs a pixel-wise class prediction.

---

#### Key Components

- **Encoder (Downsampling path)**  
  Consists of 4 convolutional blocks (`encoder1` to `encoder4`) that extract features and reduce spatial dimensions using `MaxPool2d`.

- **Bottleneck**  
  The deepest layer (`bottleneck`) captures high-level features from the compressed representation of the image.

- **Decoder (Upsampling path)**  
  Reconstructs the image using transposed convolutions (`upconv1` to `upconv4`) and combines them with encoder features using skip connections (`torch.cat`). Each combined feature map passes through a decoding block (`decoder1` to `decoder4`).

- **Final Output Layer**  
  A `1×1` convolution maps the final feature map to the desired number of classes per pixel.

---

#### Forward Pass Logic

1. The input goes through the encoder layers, progressively reducing the size and increasing feature depth.
2. The bottleneck processes the most compressed feature map.
3. The decoder upsamples the features and merges them with corresponding encoder outputs using skip connections.
4. The final output is a segmentation map with `n_classes` channels (one per class).

In [ ]:
class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=NUM_CLASSES):
        super(UNet, self).__init__()
        
        self.encoder1 = self.conv_block(n_channels, 64)
        self.encoder2 = self.conv_block(64, 128)
        self.encoder3 = self.conv_block(128, 256)
        self.encoder4 = self.conv_block(256, 512)
        
        self.bottleneck = self.conv_block(512, 1024)
        
        self.upconv4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.decoder4 = self.conv_block(1024, 512)
        
        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.decoder3 = self.conv_block(512, 256)
        
        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.decoder2 = self.conv_block(256, 128)
        
        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.decoder1 = self.conv_block(128, 64)
        
        self.final_conv = nn.Conv2d(64, n_classes, kernel_size=1)
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
    
    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool(enc1))
        enc3 = self.encoder3(self.pool(enc2))
        enc4 = self.encoder4(self.pool(enc3))
        
        bottleneck = self.bottleneck(self.pool(enc4))
        
        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.decoder4(dec4)
        
        dec3 = self.upconv3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.decoder3(dec3)
        
        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.decoder2(dec2)
        
        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.decoder1(dec1)
        
        return self.final_conv(dec1)

This function calculates the Intersection over Union (IoU) score for each class in a segmentation task. IoU measures how well the predicted segmentation matches the ground truth.

In [ ]:
def calculate_iou(pred, target, num_classes):
    ious = []    
    for cls in range(num_classes):
        pred_cls = (pred == cls)
        target_cls = (target == cls)
        intersection = (pred_cls & target_cls).sum().float()
        union = (pred_cls | target_cls).sum().float()
        
        if union == 0:
            iou = 1.0 if intersection == 0 else 0.0
        else:
            iou = (intersection / union).item()
        
        ious.append(iou)
    
    return ious

This function computes the **pixel-level accuracy** between the predicted segmentation map and the ground truth.

In [ ]:
def calculate_pixel_accuracy(pred, target):
    correct = (pred == target).sum().float()
    total = target.numel()
    return (correct / total).item()

### `train_model`: Training Loop for Semantic Segmentation

This function trains and validates a segmentation model using pixel-wise classification metrics (loss, mIoU, accuracy). It also saves the best-performing model based on validation IoU.

---

#### Parameters:
- **`model`**: The segmentation model to train (e.g., U-Net).
- **`train_loader`**: DataLoader for training dataset.
- **`val_loader`**: DataLoader for validation dataset.
- **`device`**: Device to train on (e.g., `'cuda'` or `'cpu'`).
- **`epochs`**: Total number of training epochs (default: 30).
- **`lr`**: Learning rate for the optimizer (default: 1e-3).
- **`save_path`**: File path to save the best model.

---

#### Training Setup:
- **Loss Function**: `CrossEntropyLoss` with `ignore_index=255` (ignores unlabeled pixels).
- **Optimizer**: Adam with weight decay for regularization.
- **Scheduler**: Reduces learning rate by half every 10 epochs.

---

#### Epoch Loop:
For each epoch:
1. **Training Phase**:
   - Set model to `train` mode.
   - For each batch: forward pass → compute loss → backpropagate → update weights.
   - Compute batch-level IoU and pixel accuracy.
   
2. **Validation Phase**:
   - Set model to `eval` mode.
   - No gradients are computed.
   - Similar to training, but only for evaluation.
   - Tracks loss, mean IoU, and pixel accuracy.

3. **Logging**:
   - Prints metrics for both training and validation.
   - Tracks and saves the model with the **best validation mIoU**.

4. **Learning Rate Scheduling**:
   - Updates learning rate after every epoch.

---

#### Model Saving:
- Saves model state, optimizer state, and metrics if the current model achieves the best validation mIoU so far.

---

#### Returns:
- `train_losses`: List of average training losses per epoch.
- `val_losses`: List of average validation losses per epoch.
- `val_ious`: List of validation mIoUs per epoch.

In [ ]:
def train_model(model, train_loader, val_loader, device, epochs=30, lr=1e-3, save_path="best_segmentation_model.pth"):
    print(f"Starting training for {epochs} epochs...")
    print(f"Model will be saved to: {save_path}")
    print("=" * 60)
    
    criterion = nn.CrossEntropyLoss(ignore_index=255) 
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    
    model.to(device)
    best_iou = 0.0
    train_losses = []
    val_losses = []
    val_ious = []
    
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        model.train()
        train_loss = 0.0
        train_iou = 0.0
        train_accuracy = 0.0
        
        train_progress = tqdm(train_loader, desc=f"Training", leave=False)
        
        for batch_idx, (images, masks) in enumerate(train_progress):
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            
            if isinstance(outputs, dict):  
                outputs = outputs['out']
            
            loss = criterion(outputs, masks)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
            pred = torch.argmax(outputs, dim=1)
            batch_ious = calculate_iou(pred, masks, NUM_CLASSES)
            batch_accuracy = calculate_pixel_accuracy(pred, masks)
            
            train_iou += np.mean(batch_ious)
            train_accuracy += batch_accuracy
            
            train_progress.set_postfix({
                'loss': f'{loss.item():.4f}',
                'mIoU': f'{np.mean(batch_ious):.4f}',
                'acc': f'{batch_accuracy:.4f}'
            })
        
        model.eval()
        val_loss = 0.0
        val_iou = 0.0
        val_accuracy = 0.0
        
        with torch.no_grad():
            val_progress = tqdm(val_loader, desc=f" Validation", leave=False)
            
            for images, masks in val_progress:
                images = images.to(device)
                masks = masks.to(device)
                
                outputs = model(images)
                if isinstance(outputs, dict):
                    outputs = outputs['out']
                
                loss = criterion(outputs, masks)
                val_loss += loss.item()
                
                pred = torch.argmax(outputs, dim=1)
                batch_ious = calculate_iou(pred, masks, NUM_CLASSES)
                batch_accuracy = calculate_pixel_accuracy(pred, masks)
                
                val_iou += np.mean(batch_ious)
                val_accuracy += batch_accuracy
                
                val_progress.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'mIoU': f'{np.mean(batch_ious):.4f}',
                    'acc': f'{batch_accuracy:.4f}'
                })
        
        avg_train_loss = train_loss / len(train_loader)
        avg_train_iou = train_iou / len(train_loader)
        avg_train_acc = train_accuracy / len(train_loader)
        
        avg_val_loss = val_loss / len(val_loader)
        avg_val_iou = val_iou / len(val_loader)
        avg_val_acc = val_accuracy / len(val_loader)
        
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        val_ious.append(avg_val_iou)
        
        print(f"Train - Loss: {avg_train_loss:.4f}, mIoU: {avg_train_iou:.4f}, Acc: {avg_train_acc:.4f}")
        print(f"Val   - Loss: {avg_val_loss:.4f}, mIoU: {avg_val_iou:.4f}, Acc: {avg_val_acc:.4f}")
        
        if avg_val_iou > best_iou:
            best_iou = avg_val_iou
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'best_iou': best_iou,
                'train_losses': train_losses,
                'val_losses': val_losses,
                'val_ious': val_ious
            }, save_path)
            print(f"New best model saved! mIoU: {best_iou:.4f}")
        
        scheduler.step()
        
        current_lr = scheduler.get_last_lr()[0]
        print(f" Learning rate: {current_lr:.2e}")
        print("-" * 60)
    
    print(f"Training completed!")
    print(f"Best validation mIoU: {best_iou:.4f}")
    print(f"Best model saved to: {save_path}")
    
    return train_losses, val_losses, val_ious

### `plot_training_metrics`: Visualize Loss & mIoU Over Epochs

This function plots the training and validation loss, along with the validation mean Intersection over Union (mIoU), to help monitor model performance over time.

---

#### Function Inputs:
- **`train_losses`**: List of average training losses per epoch.
- **`val_losses`**: List of average validation losses per epoch.
- **`val_ious`**: List of validation mIoUs per epoch.

---

#### What It Does:
- **Left Plot**: Shows both training and validation loss curves to check for underfitting or overfitting.
- **Right Plot**: Shows validation mIoU progression, indicating how well the model is segmenting over time.

---

#### Output:
- The plot is saved as a PNG file: `'training_metrics.png'`
- Also displayed inline in the notebook.

In [ ]:
def plot_training_metrics(train_losses, val_losses, val_ious):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    axes[0].plot(train_losses, label='Train Loss', color='blue')
    axes[0].plot(val_losses, label='Val Loss', color='red')
    axes[0].set_title('Training and Validation Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    axes[1].plot(val_ious, label='Validation mIoU', color='green')
    axes[1].set_title('Validation mIoU')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('mIoU')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.savefig('training_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(" Training metrics plot saved as 'training_metrics.png'")


### Visualizing Dataset Samples

This function helps visually inspect a few samples from the dataset by plotting:

1. **Original Image** – the input image from the dataset.
2. **Ground Truth Mask** – the corresponding segmentation mask.
3. **Overlay** – a combination of the image and its mask for better visual understanding.

This function is useful for qualitative inspection of the dataset before or after training to ensure the correctness of image-mask pairs.


In [ ]:
def visualize_dataset_samples(dataset, num_samples=4, figsize=(15, 10)):
    fig, axes = plt.subplots(num_samples, 3, figsize=figsize)
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(num_samples):
        if i >= len(dataset):
            break
            
        image, mask = dataset[i]
        
        if isinstance(image, torch.Tensor):
            img_np = image.clone()
            if img_np.dim() == 3 and img_np.shape[0] == 3:  # C, H, W
                mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
                std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
                img_np = img_np * std + mean
                img_np = torch.clamp(img_np, 0, 1)
                img_np = img_np.permute(1, 2, 0).numpy()
            else:
                img_np = image.numpy()
        else:
            img_np = np.array(image)
        
        if isinstance(mask, torch.Tensor):
            mask_np = mask.numpy()
        else:
            mask_np = np.array(mask)
        
        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title(f'Original Image {i+1}')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(mask_np, cmap='tab20', vmin=0, vmax=NUM_CLASSES-1)
        axes[i, 1].set_title(f'Ground Truth Mask {i+1}')
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(img_np)
        axes[i, 2].imshow(mask_np, alpha=0.5, cmap='tab20', vmin=0, vmax=NUM_CLASSES-1)
        axes[i, 2].set_title(f'Overlay {i+1}')
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig('dataset_samples.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("Dataset samples visualization saved as 'dataset_samples.png'")

### Visualizing Model Predictions

This function helps to qualitatively evaluate a trained segmentation model by displaying:

1. **Original Image** – the input image from the dataset.
2. **Ground Truth Mask** – the expected segmentation labels.
3. **Predicted Mask** – the mask predicted by the model.
4. **Overlay** – a transparent overlay of the prediction on the original image.

#### Function Arguments:
- `model`: Trained segmentation model.
- `dataset`: Dataset object with image-mask pairs.
- `device`: CPU or GPU (`cuda`) to perform inference.
- `num_samples`: Number of samples to visualize (default = 4).
- `figsize`: Size of the visualization figure.

#### What It Does:
- Switches the model to evaluation mode using `model.eval()`.
- Iterates over `num_samples` from the dataset.
- Passes each image through the model to get predictions.
- Uses `argmax` to convert raw logits to final class predictions.
- Denormalizes the input image for better visualization.
- Uses `matplotlib` to plot:
  - Column 1: Original Image  
  - Column 2: Ground Truth  
  - Column 3: Predicted Mask  
  - Column 4: Overlay of prediction on image  
- Saves the resulting figure as `'segmentation_predictions.png'`.

This is useful for visually checking how well the model is performing across different regions in an image.


In [ ]:
def visualize_predictions(model, dataset, device, num_samples=4, figsize=(20, 12)):
    model.eval()
    fig, axes = plt.subplots(num_samples, 4, figsize=figsize)
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(num_samples):
        if i >= len(dataset):
            break
            
        image, mask = dataset[i]
        
        with torch.no_grad():
            image_batch = image.unsqueeze(0).to(device)
            output = model(image_batch)
            
            if isinstance(output, dict):
                output = output['out']
            
            pred = torch.argmax(output, dim=1).squeeze().cpu().numpy()
        
        if isinstance(image, torch.Tensor):
            img_np = image.clone()
            if img_np.dim() == 3 and img_np.shape[0] == 3:
                mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
                std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
                img_np = img_np * std + mean
                img_np = torch.clamp(img_np, 0, 1)
                img_np = img_np.permute(1, 2, 0).numpy()
        
        mask_np = mask.cpu().numpy() if isinstance(mask, torch.Tensor) else np.array(mask)
        
        # Plot
        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title(f'Original Image {i+1}')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(mask_np, cmap='tab20', vmin=0, vmax=NUM_CLASSES-1)
        axes[i, 1].set_title(f'Ground Truth {i+1}')
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(pred, cmap='tab20', vmin=0, vmax=NUM_CLASSES-1)
        axes[i, 2].set_title(f'Prediction {i+1}')
        axes[i, 2].axis('off')
        
        axes[i, 3].imshow(img_np)
        axes[i, 3].imshow(pred, alpha=0.5, cmap='tab20', vmin=0, vmax=NUM_CLASSES-1)
        axes[i, 3].set_title(f'Prediction Overlay {i+1}')
        axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.savefig('segmentation_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("Prediction visualization saved as 'segmentation_predictions.png'")


### Function: `create_label_legend`

This function generates a visual legend for the segmentation classes used in the DocBank dataset.

- **Purpose:** To help visually identify the label-to-color mapping used during prediction visualization.
- **How it works:**
  - Iterates through all class labels defined in `DOCBANK_LABELS`.
  - Uses the `tab20` colormap to assign distinct colors to each class.
  - Constructs colored patches with corresponding class index and name.
  - Displays the legend in a clean layout and saves it as `class_legend.png`.

**Output:** A saved image file displaying class indices and their corresponding labels and colors.


In [ ]:
def create_label_legend():
    fig, ax = plt.subplots(figsize=(8, 6))

    patches = []
    labels = []
    
    for i, label in enumerate(DOCBANK_LABELS):
        color = plt.cm.tab20(i / NUM_CLASSES)
        patches.append(plt.Rectangle((0, 0), 1, 1, facecolor=color))
        labels.append(f"{i}: {label}")
    
    ax.legend(patches, labels, loc='center', fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title('Document Segmentation Classes', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('class_legend.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("Class legend saved as 'class_legend.png'")

print("Creating class legend...")
create_label_legend()

### Function: `analyze_dataset_distribution`

This function analyzes and visualizes the class distribution of pixel labels in the segmentation dataset.

- **Goal:** Understand how frequently each class appears in a sample of the dataset (up to 50 samples).
- **Steps:**
  - Initializes a dictionary to count pixel occurrences for each class.
  - Iterates through the first 50 samples (or fewer if dataset is smaller).
  - For each sample:
    - Extracts the label mask.
    - Uses `np.unique` to count how many pixels belong to each class.
  - Accumulates counts and calculates percentages for each class.
  - Prints the distribution in a tabular format.
  - Plots a bar chart showing pixel counts for all classes (log-scaled for visibility).
  - Saves the chart as `class_distribution.png`.

**Output:** A printed table and a saved bar chart visualizing class-wise pixel distribution in the dataset.


In [ ]:
def analyze_dataset_distribution(dataset):
    print("Analyzing dataset class distribution...")
    class_counts = {i: 0 for i in range(NUM_CLASSES)}
    total_pixels = 0
    
    for i in tqdm(range(min(len(dataset), 50)), desc="Analyzing samples"):
        _, mask = dataset[i]
        mask_np = mask.cpu().numpy() if isinstance(mask, torch.Tensor) else np.array(mask)
        
        unique, counts = np.unique(mask_np, return_counts=True)
        for cls, count in zip(unique, counts):
            if cls < NUM_CLASSES:
                class_counts[cls] += count
                total_pixels += count
    
    print("Class Distribution:")
    print("-" * 40)
    for cls_id, count in class_counts.items():
        percentage = (count / total_pixels) * 100 if total_pixels > 0 else 0
        print(f"{cls_id:2d}: {DOCBANK_LABELS[cls_id]:<12} {count:>8} pixels ({percentage:>5.1f}%)")
    
    # Plot distribution
    plt.figure(figsize=(12, 6))
    labels = [f"{i} {DOCBANK_LABELS[i]}" for i in range(NUM_CLASSES)]
    counts = [class_counts[i] for i in range(NUM_CLASSES)]
    
    plt.bar(range(NUM_CLASSES), counts, color=plt.cm.tab20(np.linspace(0, 1, NUM_CLASSES)))
    plt.xlabel('Class')
    plt.ylabel('Pixel Count')
    plt.title('Class Distribution in Dataset')
    plt.xticks(range(NUM_CLASSES), labels, rotation=45, ha='right')
    plt.yscale('log')  
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("Class distribution plot saved as 'class_distribution.png'")

This cell performs the full workflow for semantic segmentation training using U-Net:

1. **Setup and Configurations:**
   - Defines dataset directory, number of epochs, batch size, and image size.
   - Detects and uses GPU if available for faster training.

2. **Data Preparation:**
   - Creates data augmentations/transforms for training and validation.
   - Loads images and masks from the dataset folder.
   - Initializes a custom dataset class (`DocumentSegmentationDataset`) with training transforms.
   - Splits the dataset into 80% training and 20% validation subsets.
   - Applies validation transforms to the validation set.
   - Wraps the datasets in PyTorch `DataLoader`s for batch processing.

3. **Dataset Inspection:**
   - Visualizes a few sample images and their segmentation masks to verify data correctness.
   - Analyzes and plots class distribution to understand dataset imbalance.

4. **Model Initialization and Training:**
   - Instantiates a U-Net model configured for the number of segmentation classes.
   - Trains the model for the specified number of epochs.
   - Uses standard training loops with monitoring of loss, mean IoU, and pixel accuracy.
   - Saves the best model checkpoint based on validation IoU.

5. **Training Visualization:**
   - Plots training and validation loss curves.
   - Plots validation mean IoU over epochs to track model improvement.

6. **Model Evaluation:**
   - Visualizes model predictions on validation samples side-by-side with ground truth masks.
   - Provides qualitative insights into the model’s segmentation performance.

**NOTE:** For demonstration purposes, we are doing a quick run using 10 epochs.  
For training the segmentation model more effectively, you can try increasing the number of epochs and adjusting other hyperparameters to improve accuracy.  
Also, consider your hardware capabilities when choosing these settings.

In [ ]:
data_dir="pixel_segmentation_dataset" 
epochs=10
batch_size=4 
image_size=256
print(" Starting U-Net training")
print("=" * 60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Using device: {device}")

train_transform, val_transform = create_data_transforms(image_size)

data_path = Path(data_dir)
images_dir = data_path / "images"
masks_dir = data_path / "masks"

full_dataset = DocumentSegmentationDataset(images_dir, masks_dir, transform=train_transform)

if len(full_dataset) == 0:
    print("No data found! Please run convert_docbank_to_pixels.py first.")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f" Dataset split:")
print(f" Train: {len(train_dataset)} samples")
print(f" Val: {len(val_dataset)} samples")
print(f" Image size: {image_size}x{image_size}")
print(f" Batch size: {batch_size}")

model = UNet(n_classes=NUM_CLASSES)

print("Visualizing dataset samples...")
visualize_dataset_samples(train_dataset, num_samples=3)

print("Analyzing dataset...")
analyze_dataset_distribution(train_dataset)

print("Starting training...")
train_losses, val_losses, val_ious = train_model( model, train_loader, val_loader, device, epochs=epochs, lr=1e-3, save_path="unet_segmentation_model.pth")

print("Plotting training metrics...")
plot_training_metrics(train_losses, val_losses, val_ious)

print("Visualizing predictions...")
visualize_predictions(model, val_dataset, device, num_samples=4)

print("U-Net training completed!")